# Coverage Path Planning — Results & Analysis

This notebook analyses the training results of the curriculum-PPO agent for the Coverage Path Planning (CPP) task.

## Strategy

The original CPP environment encodes the **full visited map** (`size × size` values) in the observation, which means a model trained on a 5×5 grid cannot be applied directly to a 10×10 grid (different observation dimension).

**Solution:** `GridWorldCPPEnvV2` uses a **fixed 7×7 local window** centred on the agent instead of the full map. The observation is always 52 floats, regardless of grid size.

| Observation component | Size | Description |
|---|---|---|
| `agent_x_norm` | 1 | Agent column normalised to [0, 1] |
| `agent_y_norm` | 1 | Agent row normalised to [0, 1] |
| `coverage_ratio` | 1 | Fraction of accessible cells already visited |
| `window_flat` | 49 | 7×7 neighbourhood (0=unvisited, 0.5=visited, 1=wall) |

**Curriculum learning (two phases):**
1. **Phase 1 (5×5, 1 M steps):** agent learns the core CPP strategy in a small, fast-to-explore environment.
2. **Phase 2 (5×5 + 10×10 mix, 2 M steps):** fine-tuning on a mixed set of environments forces generalisation while preventing catastrophic forgetting of the 5×5 policy.

## 1. Imports

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from stable_baselines3 import PPO
from gymnasium_env.grid_world_cpp_v2 import GridWorldCPPEnvV2

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 2. Learning Curves

### Phase 1 — 5×5 warm-up

In [ ]:
csv_p1 = glob.glob('log/cpp_v2_5x5_*/progress.csv')
print(f'Found {len(csv_p1)} Phase-1 log(s).')

if csv_p1:
    df_p1 = pd.concat([pd.read_csv(f) for f in csv_p1], ignore_index=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.lineplot(data=df_p1, x='time/total_timesteps',
                 y='rollout/ep_rew_mean', ax=axes[0])
    axes[0].set_title('Phase 1 — Mean Reward (5×5)')
    axes[0].set_xlabel('Total timesteps')
    axes[0].set_ylabel('Mean episode reward')

    sns.lineplot(data=df_p1, x='time/total_timesteps',
                 y='rollout/ep_len_mean', ax=axes[1], color='orange')
    axes[1].set_title('Phase 1 — Mean Episode Length (5×5)')
    axes[1].set_xlabel('Total timesteps')
    axes[1].set_ylabel('Mean episode length (steps)')

    plt.tight_layout()
    plt.show()
else:
    print('No Phase-1 logs found. Run  python train_grid_world_cpp.py train  first.')

### Phase 2 — Mixed 5×5 + 10×10 fine-tune

In [ ]:
csv_p2 = glob.glob('log/cpp_v2_mixed_*/progress.csv')
print(f'Found {len(csv_p2)} Phase-2 log(s).')

if csv_p2:
    df_p2 = pd.concat([pd.read_csv(f) for f in csv_p2], ignore_index=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.lineplot(data=df_p2, x='time/total_timesteps',
                 y='rollout/ep_rew_mean', ax=axes[0])
    axes[0].set_title('Phase 2 — Mean Reward (5×5 + 10×10 mix)')
    axes[0].set_xlabel('Total timesteps')
    axes[0].set_ylabel('Mean episode reward')

    sns.lineplot(data=df_p2, x='time/total_timesteps',
                 y='rollout/ep_len_mean', ax=axes[1], color='orange')
    axes[1].set_title('Phase 2 — Mean Episode Length')
    axes[1].set_xlabel('Total timesteps')
    axes[1].set_ylabel('Mean episode length (steps)')

    plt.tight_layout()
    plt.show()
else:
    print('No Phase-2 logs found. Run  python train_grid_world_cpp.py train  first.')

## 3. Generalisation Evaluation

Load the Phase-2 model and evaluate it on 100 episodes for each grid size.

In [ ]:
import os

# Auto-detect the most recent Phase-2 model
models = sorted(glob.glob('data/cpp_v2_mixed_*.zip'))
if not models:
    raise FileNotFoundError('No Phase-2 model found in data/. '
                            'Run  python train_grid_world_cpp.py train  first.')

model_path = models[-1]
print(f'Loading model: {model_path}')
model = PPO.load(model_path)

In [ ]:
ENV_CONFIGS = {
    5:  {'obs_quantity': 3,  'max_steps': 200},
    10: {'obs_quantity': 10, 'max_steps': 400},
    20: {'obs_quantity': 40, 'max_steps': 1000},
}

N_EVAL = 100
results = {}

for size, cfg in ENV_CONFIGS.items():
    env = GridWorldCPPEnvV2(size=size, **cfg)
    coverages, steps_list = [], []

    for ep in range(N_EVAL):
        obs, info = env.reset(seed=ep)
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, _, terminated, truncated, info = env.step(int(action))
            done = terminated or truncated
        coverages.append(info['coverage_ratio'] * 100)
        steps_list.append(info['steps'])

    env.close()
    results[size] = {'coverage': coverages, 'steps': steps_list}
    full = sum(c >= 99.0 for c in coverages)
    print(f'{size:2d}×{size:<2d}  mean={np.mean(coverages):.1f}%  '
          f'std={np.std(coverages):.1f}%  '
          f'full={full}/{N_EVAL}')

### Coverage distribution across grid sizes

In [ ]:
rows = []
for size, data in results.items():
    for cov, st in zip(data['coverage'], data['steps']):
        rows.append({'Grid size': f'{size}×{size}', 'Coverage (%)': cov, 'Steps': st})
df_eval = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_eval, x='Grid size', y='Coverage (%)', ax=axes[0])
axes[0].axhline(99, color='red', linestyle='--', linewidth=1, label='99% threshold')
axes[0].set_title('Coverage distribution by grid size')
axes[0].set_ylim(0, 105)
axes[0].legend()

sns.boxplot(data=df_eval, x='Grid size', y='Steps', ax=axes[1], color='salmon')
axes[1].set_title('Episode length by grid size')

plt.tight_layout()
plt.show()

### Summary table

In [ ]:
summary_rows = []
for size, data in results.items():
    covs = data['coverage']
    sts  = data['steps']
    full = sum(c >= 99.0 for c in covs)
    summary_rows.append({
        'Grid': f'{size}×{size}',
        'Mean cov (%)': round(np.mean(covs), 1),
        'Std (%)': round(np.std(covs), 1),
        'Min (%)': round(np.min(covs), 1),
        'Max (%)': round(np.max(covs), 1),
        'Full coverage': f'{full}/{N_EVAL}',
        'Mean steps': round(np.mean(sts), 1),
    })

pd.DataFrame(summary_rows).set_index('Grid')

## 4. Discussion

### Why the local window enables generalisation
The agent never observes the global state of the grid — it only sees what is immediately around it. This forces it to learn a **local exploration strategy** (e.g., prefer unvisited neighbours, avoid revisiting cells) that is valid regardless of the total grid size. The normalised position and coverage ratio give the agent just enough global context to adjust its behaviour as the episode progresses.

### Curriculum learning vs. direct training on 10×10
Training directly on 10×10 from scratch is harder because the sparse reward signal (the completion bonus +10 is rarely seen early in training). Starting on 5×5 lets the agent discover the completion bonus quickly, shaping the policy before moving to the harder task.

### Limitations
- The local window (7×7) can miss information about remote unvisited regions, leading the agent to revisit areas it has already covered when the remaining unvisited cells are far away.
- Performance on 20×20 is zero-shot (the model was never trained on that size), so some degradation is expected.
- The obstacle density is fixed per grid size; very different densities may require additional fine-tuning.

### Possible improvements
- **Graph-based state:** encode the remaining frontier cells as a graph and use a GNN policy.
- **Larger window / recurrent policy:** a GRU or LSTM could accumulate a mental map without encoding it explicitly.
- **Reward shaping:** penalise repeated revisiting of the same cell more aggressively.